In [1]:
import os
import hashlib
from pathlib import Path
from typing import List, Dict

import numpy as np
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

d:\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. PDF INGESTION

In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("D:\Project\RagQnA\data")

<>:33: SyntaxWarning: invalid escape sequence '\P'
<>:33: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Aishwarya\AppData\Local\Temp\ipykernel_15400\1217676137.py:33: SyntaxWarning: invalid escape sequence '\P'
  all_pdf_documents = process_all_pdfs("D:\Project\RagQnA\data")


Found 2 PDF files to process

Processing: 2bf1f0e9f04e6fb4f8fef35e82c42aa5 1.pdf
  ✓ Loaded 21 pages

Processing: bankingPolicy.pdf
  ✓ Loaded 24 pages

Total documents loaded: 45


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'iTextSharp™ 5.5.13.1 ©2000-2019 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2023-08-12T02:13:03+05:30', 'moddate': '2023-08-12T02:14:35+05:30', 'source': 'D:\\Project\\RagQnA\\data\\2bf1f0e9f04e6fb4f8fef35e82c42aa5 1.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': '2bf1f0e9f04e6fb4f8fef35e82c42aa5 1.pdf', 'file_type': 'pdf'}, page_content='THE DIGITAL PERSONAL DATA PROTECTION ACT, 2023\n(NO. 22 OF 2023)\n[11th August, 2023.]\nAn Act to provide for the processing of digital personal data in a manner that\nrecognises both the right of individuals to protect their personal data and the\nneed to process such personal data for lawful purposes and for matters\nconnected therewith or incidental thereto.\nBE it enacted by Parliament in the Seventy-fourth Year of the Republic of India as\nfollows:––\nCHAPTER I\nPRELIMINARY\n1. (1) This Act may be called the Digital Personal Data Protection Act, 2023.\n(2)  It shall 

2. CHUNKING (✔ FIXED OVERLAP)

In [4]:
### Text splitting get into chunks

def split_documents(documents):
    """
    Optimal chunking for legal / policy PDFs
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,        # ✔ Optimal size
        chunk_overlap=100,     # ✔ Reduced overlap (important fix)
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    return chunks


In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Created 194 chunks


[Document(metadata={'producer': 'iTextSharp™ 5.5.13.1 ©2000-2019 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2023-08-12T02:13:03+05:30', 'moddate': '2023-08-12T02:14:35+05:30', 'source': 'D:\\Project\\RagQnA\\data\\2bf1f0e9f04e6fb4f8fef35e82c42aa5 1.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': '2bf1f0e9f04e6fb4f8fef35e82c42aa5 1.pdf', 'file_type': 'pdf'}, page_content='THE DIGITAL PERSONAL DATA PROTECTION ACT, 2023\n(NO. 22 OF 2023)\n[11th August, 2023.]\nAn Act to provide for the processing of digital personal data in a manner that\nrecognises both the right of individuals to protect their personal data and the\nneed to process such personal data for lawful purposes and for matters\nconnected therewith or incidental thereto.\nBE it enacted by Parliament in the Seventy-fourth Year of the Republic of India as\nfollows:––\nCHAPTER I\nPRELIMINARY\n1. (1) This Act may be called the Digital Personal Data Protection Act, 2023.\n(2)  It shall 

3. EMBEDDING MANAGER

In [6]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        print(f"Embedding model loaded: {model_name}")

    def generate(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, show_progress_bar=True)


4. VECTOR STORE (✔ DEDUP SAFE)

In [7]:
class VectorStore:
    def __init__(self, persist_directory="./vector_store"):
        os.makedirs(persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collection = self.client.get_or_create_collection("pdf_documents")

    def add_documents(self, documents, embeddings):
        existing_ids = set(self.collection.get()["ids"])

        ids, texts, metadatas, embed_list = [], [], [], []

        for doc, emb in zip(documents, embeddings):
            # ✔ CONTENT HASH FOR DEDUP
            content_hash = hashlib.md5(
                doc.page_content.strip().encode("utf-8")
            ).hexdigest()

            if content_hash in existing_ids:
                continue

            ids.append(content_hash)
            texts.append(doc.page_content)
            metadatas.append(doc.metadata)
            embed_list.append(emb.tolist())

        if ids:
            self.collection.add(
                ids=ids,
                documents=texts,
                metadatas=metadatas,
                embeddings=embed_list
            )

        print(f"Added {len(ids)} unique chunks to vector DB")


5. PRODUCTION-LEAN RETRIEVER

In [8]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedder: EmbeddingManager):
        self.vector_store = vector_store
        self.embedder = embedder

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.4
    ) -> List[Dict]:

        print(f"\nQuery: {query}")

        query_embedding = self.embedder.generate([query])[0]

        # ✔ Over-retrieve
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=15
        )

        final_results = []
        seen_texts = set()

        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ):
            similarity = 1 - dist

            if similarity < score_threshold:
                continue

            normalized = doc.strip().lower()
            if normalized in seen_texts:
                continue

            seen_texts.add(normalized)

            final_results.append({
                "content": doc,
                "metadata": meta,
                "similarity_score": round(similarity, 3)
            })

            if len(final_results) == top_k:
                break

        return final_results


In [9]:
if __name__ == "__main__":

    PDF_DIRECTORY = r"D:\Project\RagQnA\data"

    docs = process_all_pdfs(PDF_DIRECTORY)
    chunks = split_documents(docs)

    embedder = EmbeddingManager()
    embeddings = embedder.generate([c.page_content for c in chunks])

    vectorstore = VectorStore()
    vectorstore.add_documents(chunks, embeddings)

    retriever = RAGRetriever(vectorstore, embedder)

    query = "in order, can the company continue processing the data?"
    results = retriever.retrieve(query)

    for i, r in enumerate(results, 1):
        print(f"\n--- Result {i} ---")
        print(r["content"][:500])
        print("Score:", r["similarity_score"])
        print("Metadata:", r["metadata"])

Found 2 PDF files to process

Processing: 2bf1f0e9f04e6fb4f8fef35e82c42aa5 1.pdf
  ✓ Loaded 21 pages

Processing: bankingPolicy.pdf
  ✓ Loaded 24 pages

Total documents loaded: 45
Created 194 chunks
Embedding model loaded: all-MiniLM-L6-v2


Batches:   0%|          | 0/7 [00:00<?, ?it/s]d:\Project\.venv\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Batches: 100%|██████████| 7/7 [00:06<00:00,  1.14it/s]


Added 0 unique chunks to vector DB

Query: in order, can the company continue processing the data?


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.24it/s]
